# TekDex HR Policies RAG Chatbot

## Project Overview

In this project, I have built an end-to-end Retrieval-Augmented Generation (RAG) chatbot for TekDex company HR policies. I designed this system to allow employees to query company policy documents (`TekDex_Policies_docs`) and receive accurate, context-grounded answers citing official policy documentation.

**RAG Pipeline Architecture:**
1. **Document Ingestion:** Parses company policy documents in PDF, DOCX, and TXT formats with full metadata tracking.
2. **Text Splitting & Chunking:** Recursive character splitting with contextual overlap to retain clause integrity.
3. **Vector Embeddings & Storage:** CPU-efficient dense embeddings using `all-MiniLM-L6-v2` indexed in a local Chroma vector database.
4. **Grounded Reasoning:** Google Gemini LLM conditioned with a strict prompt to answer solely from retrieved policy context and cite sources.
5. **Interactive UI:** Web-based conversational chatbot interface built using Gradio.

## 1. Install required packages for this RAG-Company project

In this step, I install the required packages and dependencies for this RAG-Company project, including LangChain, ChromaDB, HuggingFace sentence-transformers, Google Generative AI, document loaders, and Gradio for the chat interface.

In [1]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma langchain-huggingface sentence-transformers chromadb pypdf docx2txt python-dotenv google-api-core gradio

Note: you may need to restart the kernel to use updated packages.


## 2. Load the Gemini API key from `.env`

In this step, I load the Gemini API key securely from the local `.env` environment file. This ensures API credentials remain protected and verifies model authentication before proceeding.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Search for .env from the notebook's current folder and its parent folders.
load_dotenv(Path.cwd() / ".env")
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError(
        "Gemini API key not found. Add GOOGLE_API_KEY=... to a .env file beside the notebook."
    )

print("Gemini API key loaded successfully (the key itself is not displayed).")

Gemini API key loaded successfully (the key itself is not displayed).


## 3. Load documents from the policy folder

In this step, I load and extract text from the company HR policy documents. The loader parses PDF, DOCX, and TXT files while attaching document-level metadata so the chatbot can cite exact source files when answering queries.

In [3]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

# Locate the policy documents folder.
possible_folders = [Path("TekDex_Policies_docs"), Path("TekDex_Policies_doc")]
policy_folder = next((folder for folder in possible_folders if folder.is_dir()), None)

if policy_folder is None:
    raise FileNotFoundError(
        "Policy folder not found. Put TekDex_Policies_docs (or TekDex_Policies_doc) beside the notebook."
    )

print(f"Reading policy files from: {policy_folder.resolve()}")

documents = []

for file_path in sorted(policy_folder.rglob("*")):
    if not file_path.is_file():
        continue

    extension = file_path.suffix.lower()
    if extension == ".pdf":
        file_documents = PyPDFLoader(str(file_path)).load()
    elif extension == ".docx":
        file_documents = Docx2txtLoader(str(file_path)).load()
    elif extension == ".txt":
        file_documents = TextLoader(str(file_path), autodetect_encoding=True).load()
    else:
        continue  # Ignore files that are not one of the supported formats.

    # Keep a short, stable source name for citations in chatbot answers.
    for document in file_documents:
        document.metadata["source"] = file_path.name
    documents.extend(file_documents)
    print(f"Loaded: {file_path.name} ({len(file_documents)} page/document item(s))")

if not documents:
    raise ValueError("No readable PDF, DOCX, or TXT policy documents were found.")

print(f"\nLoaded {len(documents)} page/document item(s) from {policy_folder}.")

C:\Users\BizTech\AppData\Local\Temp\ipykernel_19040\2719648565.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


Reading policy files from: C:\Users\BizTech\Desktop\Rag Project\TekDex_Policies_docs
Loaded: TDX_Code Of Conduct Policy_Jan 2026..pdf (6 page/document item(s))
Loaded: TDX_Company Employee Benefits Policy-July 2026.pdf (11 page/document item(s))
Loaded: TDX_Employee Exit, Clearance & Full & Final (FnF) Settlement Policy-April 2026.pdf (8 page/document item(s))
Loaded: TDX_Employee Loan Policy-September 2026...pdf (9 page/document item(s))
Loaded: TDX_Leave Application Procedure Document_Jan 2026..pdf (9 page/document item(s))
Loaded: TDX_Leave Management Policy Document_Jan 2026..pdf (11 page/document item(s))
Loaded: TDX_Performance Appraisal & Increment Policy-April 2026.pdf (10 page/document item(s))
Loaded: TDX_Probation Evaluation & Completion  Policy Document_2026.pdf (5 page/document item(s))
Loaded: TDX_Working Hours Policy-Jan 2026..pdf (6 page/document item(s))

Loaded 75 page/document item(s) from TekDex_Policies_docs.


## 4. Split documents into chunks

To optimize vector retrieval and stay within LLM context constraints, I split the policy documents into chunks of 1,000 characters with 150 characters of overlap using `RecursiveCharacterTextSplitter`. The overlap maintains contextual continuity across chunk boundaries.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = text_splitter.split_documents(documents)

if not chunks:
    raise ValueError("Document loading succeeded, but no text chunks were created.")

print(f"Created {len(chunks)} text chunks.")
print("Example chunk:")
print(chunks[0].page_content[:500])

Created 168 text chunks.
Example chunk:
Code
 
of
 
Conduct,
 
Discipline
 
&
 
Anti-
 
Harrassment
  
 
Policy
 
Document
 
 
Document
 
Code:
 
TDX-PO-COC-01
 
 
Version
 
No.:
 
1.0
 
 
09
 
January
 
2026
 
 
Prepared
 
by:
 
Human
 
Resource
 
Department


## 5. Create embeddings and store them in Chroma

For semantic search, I generate vector embeddings using the compact, CPU-efficient `sentence-transformers/all-MiniLM-L6-v2` model. I then index these embedded chunks into a local Chroma vector database (`chroma_db`) under a dedicated collection, enabling fast similarity search.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Local HuggingFace sentence embedding model running on CPU.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

database_folder = Path("chroma_db")
collection_name = "tekdex_hr_policies"

# Recreate collection from current policy chunks.
old_vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(database_folder),
)
old_vector_store.delete_collection()

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(database_folder),
)

print(f"Indexed {len(chunks)} chunks in Chroma at: {database_folder.resolve()}")

c:\Users\BizTech\Desktop\Rag Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5018.86it/s]


Indexed 168 chunks in Chroma at: C:\Users\BizTech\Desktop\Rag Project\chroma_db


## 6. Build the RAG chain using Gemini

Here, I build the complete RAG query pipeline. When a user submits a question, Chroma retrieves the 4 most relevant policy chunks. A customized prompt directs Google Gemini to answer strictly using the retrieved policy context and cite the source policy file names.

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai.chat_models import ChatGoogleGenerativeAIError, GoogleAPIError
from langchain_core.prompts import ChatPromptTemplate

# Flash-Lite is Google's lower-latency, cost-effective option for simple policy Q&A.
chat_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=google_api_key,
    temperature=0,
    request_timeout=30,
    retries=1,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant answering questions about TekDex HR policies. "
            "Answer using only the policy context below. Do not invent rules, dates, "
            "eligibility, or exceptions. If the context does not contain enough information, "
            "say that you could not find the answer in the available policy documents and "
            "suggest checking with HR. Cite the supporting policy filename(s) in square brackets, "
            "for example [Leave Management Policy.pdf]. Be clear and concise.\n\n"
            "Policy context:\n{context}"
        ),
        ("human", "{question}"),
    ]
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def answer_question(question):
    """Find policy passages and ask Gemini to answer from those passages only."""
    question = question.strip()
    if not question:
        return "Please enter a question about the HR policies."
    
    matching_chunks = retriever.invoke(question)
    if not matching_chunks:
        return "I could not find relevant policy text. Please check with HR."

    context_parts = []
    for chunk in matching_chunks:
        source_name = chunk.metadata.get("source", "Unknown policy file")
        page_number = chunk.metadata.get("page")
        page_note = f", page {page_number + 1}" if isinstance(page_number, int) else ""
        context_parts.append(
            f"[Source: {source_name}{page_note}]\n{chunk.page_content}"
        )

    formatted_prompt = prompt.invoke(
        {"context": "\n\n---\n\n".join(context_parts), "question": question}
    )
    try:
        response = chat_model.invoke(formatted_prompt)
    except GoogleAPIError as error:
        return f"Gemini is temporarily busy or unavailable. Wait a minute and try again. Details: {error}"
    except ChatGoogleGenerativeAIError as error:
        return f"Gemini could not answer the question. Check that your API key is valid, the model is available, and your quota is not exceeded. Details: {error}"
    return response.content

print(answer_question("What should I do to apply for leave?"))

c:\Users\BizTech\Desktop\Rag Project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'To apply for leave, you must follow the notice periods and conditions specific to the type of leave you are requesting:\n\n* **Annual Leave:** Must be applied for 15 days in advance and approved by your reporting manager or HR department [TDX_Leave Management Policy Document_Jan 2026..pdf, page 4]. \n* **Umrah Leave:** Must be applied for 15 days prior to the leave, along with supporting documents (Visa, Hotel reservations, and Tickets reservations) for approval [TDX_Leave Management Policy Document_Jan 2026..pdf, page 7, 10].\n* **Hajj Leave:** Must be applied for 1 month prior to the leave, subject to approval and submission of Visa and Hotel/ticket documents [TDX_Leave Management Policy Document_Jan 2026..pdf, page 7, 10].\n* **Maternity Leave:** An application must be submitted 3 months prior for approval [TDX_Leave Management Policy Document_Jan 2026..pdf, page 6].\n* **Paternity Leave:** An application must be submitted 15 days prior for approval [TDX_L

## 7. Create the Gradio chatbot interface

To provide an intuitive interface for employees, I built an interactive web-based chat assistant using Gradio. It connects directly to the RAG pipeline, providing conversational policy Q&A with predefined sample queries and source-cited responses.

In [19]:
import gradio as gr

def chat(message, history):
    """Handle one user message from the Gradio chat window."""
    return answer_question(message)

chatbot = gr.ChatInterface(
    fn=chat,
    title="TekDex HR Policy Assistant",
    description="Ask a question about the available TekDex HR policy documents. Answers include policy citations.",
    analytics_enabled=False,
    examples=[
        "How do I apply for leave?",
        "What are the working hours?",
        "What happens during probation evaluation?",
        "What steps are part of employee clearance?",
    ],
)

print("Gradio chatbot interface is ready.")

Gradio chatbot is ready. Run the next cell to open it.


## 8. Launch and test the chatbot interface

In this final step, I launch the Gradio application to test and interact with the deployed HR policy assistant. The interface allows querying various HR topics—such as leave policies, probation guidelines, working hours, and clearance procedures—with grounded answers and direct source attribution.

> **Note:** This assistant serves as a reference tool grounded in official TekDex HR documentation. For ambiguous scenarios or formal requests, employees are advised to consult HR directly.

In [ ]:
chatbot.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


c:\Users\BizTech\Desktop\Rag Project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\BizTech\Desktop\Rag Project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
